# Diagnose MCP OAuth configuration drift without collecting credentials

Draft contribution by **Sophia Cole / ColeWorld**. MIT licensed; see LICENSE. This contribution has not been accepted or endorsed by OpenAI.

A connector can have plausible settings and still fail before an agent can call a tool. This recipe narrows the investigation by comparing three kinds of deliberately selected **public** evidence: what was configured, what discovery advertised, and what the client actually used.

You will build a deterministic preflight report, see common mismatch patterns, preserve missing evidence as unknown, and avoid falsely claiming that matching settings prove authentication.

**Offline by design:** synthetic `.test` domains only, no credentials, HTTP requests, OAuth exchanges, model calls, or live MCP connections. The core helper requires only Python 3.10+. Run from this notebook's directory so its adjacent modules can be imported.

## 1. The evidence contract

Never paste OAuth traces, HTTP headers, tokens, secrets, cookies, codes, PKCE verifiers, or browser callback responses into this notebook. Select only the documented public fields before invoking the helper. The strict schema rejects unexpected/nested fields and recognizable unsafe forms; it is **not a general secret detector or trace sanitizer**. An arbitrary secret could still resemble a scope name or URL path.

- `configured` records intended values for one attempt.
- `discovered` records selected public metadata fields. `issuer` belongs to authorization-server metadata; `resource` belongs to protected-resource metadata.
- `observed` records public values actually used or reported in that attempt. In particular, redirect fields are the `redirect_uri` parameters, not the returned browser URL.

Omit anything not collected. An omitted value means unknown; `null` is invalid. All reports are value-free, including errors. See README.md for the complete schema, restricted URL profile, and comparison rules.

In [ ]:
from copy import deepcopy
import json

from oauth_doctor import diagnose, diagnose_json
from safe_fixtures import matching_evidence


def find_check(report, check_id):
    return next(item for item in report["checks"] if item["id"] == check_id)


def show(report):
    print("Result:", report["status"])
    print("Authentication:", report["authentication"], "| Tool access:", report["tool_access"])
    for item in report["checks"]:
        if item["status"] != "match":
            print(item["id"], "->", item["status"], "|", ", ".join(item["fields"]))
            print(" ", item["explanation"])
            print(" ", item["action"])
    for error in report["validation_errors"]:
        print(error["code"], "|", error["field"], "|", error["action"])


## 2. A complete, matching synthetic record

The fixture contains the same synthetic issuer, resource, endpoints, and effective callback across its applicable sections, plus public-client mode and two fictional scopes. Even when all 18 checks match, the strongest conclusion is **settings consistency in supplied evidence**. No source is independently authenticated and no MCP tool is called.

In [ ]:
baseline = matching_evidence()
report = diagnose(baseline)
assert report["status"] == "settings_match"
assert report["authentication"] == "not_verified"
assert report["tool_access"] == "not_verified"
assert len(report["checks"]) == 18
show(report)


## 3. Reproduce drift without a login

Introduce three realistic configuration differences into a copy: the token request points at a different resource, the token exchange uses a different effective callback, and the effective client type differs from its configuration. The report names the relevant fields without reproducing their values.

Issuer and callback comparisons are exact. Resource comparison tolerates only scheme/host case; it does not erase path case, trailing slashes, explicit ports, percent escapes, or dot segments. A difference is evidence to investigate, not permission to automatically rewrite a live configuration.

In [ ]:
drift = deepcopy(baseline)
drift["observed"]["token_resource"] = "https://other.example.test/mcp"
drift["observed"]["token_redirect_uri"] = "https://app.example.test/callback/"
drift["observed"]["client_type"] = "confidential"
drift_report = diagnose(drift)
assert drift_report["status"] == "mismatch"
for check_id in ["resource.token", "redirect.token", "client.type"]:
    assert find_check(drift_report, check_id)["status"] == "mismatch"
show(drift_report)


## 4. Scope sets, not text formatting

Scope names are case-sensitive; ordering and repeated names do not add permissions. Compare scope sets rather than order. This example also demonstrates MCP's scheme/host case tolerance for resource identifiers. Neither rule is applied to the issuer or effective callback.

A granted scope set can differ from the requested set under provider policy. Such a difference is a diagnostic mismatch, not proof that authentication failed.

In [ ]:
reordered = deepcopy(baseline)
reordered["observed"]["requested_scopes"] = ["files:write", "files:read", "files:read"]
reordered["observed"]["granted_scopes"] = ["files:write", "files:read"]
reordered["observed"]["authorization_resource"] = "HTTPS://MCP.EXAMPLE.TEST/mcp"
assert diagnose(reordered)["status"] == "settings_match"
print("Scope ordering/duplicates and resource scheme/host case do not create false drift.")


## 5. Preserve missing evidence

Delete the recorded grant scopes. Do not copy the request into the grant field, assume a successful grant, or decode an unverified token to fill the gap. The recipe deliberately does not infer missing metadata defaults either: an omitted copied field might simply mean it was never collected.

In [ ]:
incomplete = deepcopy(baseline)
del incomplete["observed"]["granted_scopes"]
incomplete_report = diagnose(incomplete)
assert incomplete_report["status"] == "evidence_incomplete"
assert find_check(incomplete_report, "scope.granted")["status"] == "unknown"
show(incomplete_report)


## 6. An advertisement is not an exhaustive authorization policy

An authorization server can omit supported scopes from its metadata. A configured scope absent from `scopes_supported` therefore requires review, rather than a definitive unsupported-scope verdict. This fixture does not contain or evaluate a live scope challenge.

In [ ]:
advertisement = deepcopy(baseline)
advertisement["discovered"]["scopes_supported"] = ["files:read"]
advertisement_report = diagnose(advertisement)
assert advertisement_report["status"] == "review_required"
assert find_check(advertisement_report, "scope.advertised")["status"] == "review"
show(advertisement_report)


## 7. Reject unsupported input instead of sanitizing arbitrary logs

The following intentionally invalid document contains only a synthetic sentinel, not a real credential. Unknown fields reject the whole input before comparisons run. Error reports never echo raw values, unknown field names, or parser exception details.

This protects the report surface; it does not make collecting secrets acceptable. Validation cannot reliably distinguish all opaque secrets from public-looking values. Keep actual credentials entirely outside the input contract and outside notebooks.

In [ ]:
bad = {"observed": {"raw_headers": {"Authorization": "SYNTHETIC_DO_NOT_ECHO"}}}
rejected = diagnose(bad)
assert rejected["status"] == "invalid_input"
assert rejected["checks"] == []
assert "SYNTHETIC_DO_NOT_ECHO" not in json.dumps(rejected)
assert "Authorization" not in json.dumps(rejected)
assert diagnose_json('{"observed": {}, "observed": {}}')["status"] == "invalid_input"
show(rejected)


## 8. Recheck a corrected synthetic record

Return the three modified fields to the intended synthetic values and rerun. This proves only that the supplied settings now agree. In a real investigation, a human would review the original sources before changing anything, then separately authorize and perform any live runtime verification.

An agent can consume the returned fixed check IDs, statuses, and field paths to explain the next investigation step without receiving an OAuth trace. Keep the validation in application code, not only in a model prompt. This notebook intentionally has no model invocation or live tool-testing branch.

In [ ]:
corrected = deepcopy(drift)
for field in ["token_resource", "token_redirect_uri", "client_type"]:
    corrected["observed"][field] = baseline["observed"][field]
corrected_report = diagnose(corrected)
assert corrected_report["status"] == "settings_match"
assert corrected_report["authentication"] == "not_verified"
assert corrected_report["tool_access"] == "not_verified"
show(corrected_report)


## Verification and boundaries

Run `python -m unittest -v test_oauth_doctor` for behavioral tests and `python validate_notebook.py` for nbformat schema validation plus sequential Python-cell execution. The latter uses one shared namespace with socket/DNS stubs; it is **not a full Jupyter kernel execution test or an execution sandbox**. Source notebook outputs are intentionally empty.

There are no claims about TLS, DNS, metadata provenance, client registration, state/PKCE, signatures, token audience or expiry, consent, authentication, MCP initialization, or runtime tool authorization. These require different evidence and are outside this recipe.

## Primary references

Checked 2026-10-23. The pinned MCP revision keeps protocol assumptions explicit.

- [MCP authorization, 2025-11-25](https://modelcontextprotocol.io/specification/2025-11-25/basic/authorization): resource/callback and scope guidance.
- [RFC 8414, sections 2, 3.3, 4](https://www.rfc-editor.org/rfc/rfc8414.html#section-3.3): issuer equality, metadata and string comparisons.
- [RFC 8707, section 2](https://www.rfc-editor.org/rfc/rfc8707.html#section-2): resource identifiers.
- [RFC 6749, sections 3.1.2.3 and 3.3](https://www.rfc-editor.org/rfc/rfc6749.html#section-3.3): callback and scope comparisons.
- [RFC 8252, section 7.3](https://www.rfc-editor.org/rfc/rfc8252.html#section-7.3): native-app loopback callback registration. The helper compares concrete per-attempt callbacks, not registration templates.

See README.md for the exact input schema, deliberate exclusions, and report precedence; see VERIFICATION.md for test evidence and unverified layers.